# E0 — Calibration tables (Step A)

Questions (`IMPLEMENTATION_PLAN.md` §14.1 E0):
1. Does Ada-ef hit its declared recall at idle (`calib_mean_recall_at_pref` within ±0.02 of the target)?
2. What do the price lists look like: recall r̄_g(e) and work c̄_g(e) per difficulty group, loss ratios ℓ, group shares p_g?
3. How much of the work is the Ada-ef probe (`probe_ndis / ndis`)? This bounds the budget's control range [B_min, B_max] (plan finding 4).

**Inputs** (written by `laef_calibrate`, plan §9): `data/artifacts/<dataset>/tables/price_list.json` and `tables/calibration_sweep.csv` (columns `query, score, ef_pref, ef, recall, ndis, probe_ndis, service_us, is_pref`). Datasets without these files are skipped.

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:  # walk up to Implementation/
    ROOT = ROOT.parent
ARTIFACTS = ROOT / 'data' / 'artifacts'
FIG = ROOT / 'results' / 'figures' / 'e0_calibration'
FIG.mkdir(parents=True, exist_ok=True)

DATASETS = ['glove-100-angular', 'deep-image-96-angular', 'dbpedia-openai-1m', 'cluster-zipf-100d', 'coco-i2i-512-angular', 'coco-t2i-512-angular', 'lastfm-64-dot', 'synthetic-small']
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

def load_calibration(ds):
    t = ARTIFACTS / ds / 'tables'
    if not (t / 'price_list.json').exists() or not (t / 'calibration_sweep.csv').exists():
        return None
    pl = json.loads((t / 'price_list.json').read_text())
    sweep = pd.read_csv(t / 'calibration_sweep.csv')
    sweep['is_pref'] = sweep['is_pref'].astype(bool)
    return pl, sweep

def group_of(score, groups):
    s = int(np.clip(int(score), 0, 100))  # Ada-ef key: int(score)
    for g in groups:
        if g['score_lo'] <= s <= g['score_hi']:
            return g['id']
    return groups[-1]['id']

cal = {ds: c for ds in DATASETS if (c := load_calibration(ds)) is not None}
print('datasets with calibration tables:', list(cal) or 'none (run scripts/run_pipeline.sh first)')

## 1. Summary table

`floor/nominal` close to 1 means the probe already does most of the work, so cutting ef saves little (plan finding 4; lever: `adaef.statics_length`).

In [ ]:
rows = []
for ds, (pl, sweep) in cal.items():
    pref = sweep[sweep.is_pref]
    rec = pref['recall'].to_numpy()
    rows.append({
        'dataset': ds,
        'k': pl['k'],
        'target': pl['target_recall'],
        'calib queries': len(pref),
        'groups': len(pl['groups']),
        'ladder': pl['ladder'],
        'static_ef_for_target': pl['static_ef_for_target'],
        'mean recall @ ef_pref': pl['calib_mean_recall_at_pref'],
        'P5 recall': np.percentile(rec, 5) if len(rec) else np.nan,
        'P1 recall': np.percentile(rec, 1) if len(rec) else np.nan,
        'within ±0.02': abs(pl['calib_mean_recall_at_pref'] - pl['target_recall']) <= 0.02,
        'floor work c(k)': pl['floor_work_mean'],
        'nominal work': pl['nominal_work_mean'],
        'floor/nominal': pl['floor_work_mean'] / pl['nominal_work_mean'],
        'probe/ndis @ pref': (pref['probe_ndis'] / pref['ndis'].clip(lower=1)).mean(),
        'mean ef_pref': pref['ef_pref'].mean(),
    })
summary = pd.DataFrame(rows).set_index('dataset') if rows else pd.DataFrame()
summary.to_csv(FIG / 'e0_summary.csv')
summary.round(3) if rows else 'no data'

## 2. Recall and work curves per difficulty group

Lines are the kept (concave-envelope) rungs from `price_list.json`; markers are the raw group means recomputed from the sweep, so rungs dropped by the envelope are visible.

In [ ]:
def raw_group_means(pl, sweep):
    s = sweep[~sweep.is_pref].copy()
    s['group'] = [group_of(x, pl['groups']) for x in s['score']]
    return s.groupby(['group', 'ef'])[['recall', 'ndis']].mean().reset_index()

for ds, (pl, sweep) in cal.items():
    raw = raw_group_means(pl, sweep)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    cmap = plt.get_cmap('viridis', max(2, len(pl['groups'])))
    for g in pl['groups']:
        c = cmap(g['id'])
        lab = f"g{g['id']} score {g['score_lo']}-{g['score_hi']} (p={g['p']:.2f})"
        r = raw[raw.group == g['id']]
        ax[0].plot(g['ladder'], g['recall'], '-', color=c, label=lab)
        ax[0].plot(r['ef'], r['recall'], 'o', color=c, ms=3, alpha=0.6)
        ax[1].plot(g['ladder'], g['work'], '-', color=c)
        ax[1].plot(r['ef'], r['ndis'], 'o', color=c, ms=3, alpha=0.6)
        ax[2].plot(g['work'], g['recall'], '-o', color=c, ms=3)
        ax[2].axvline(g['work'][0], color=c, lw=0.5, ls=':')
    ax[0].axhline(pl['target_recall'], color='k', ls='--', lw=1, label='target')
    ax[0].set(xscale='log', xlabel='ef', ylabel='mean recall r̄_g(e)', title=f'{ds}: recall')
    ax[1].set(xscale='log', yscale='log', xlabel='ef', ylabel='mean distance computations c̄_g(e)', title='work')
    ax[2].set(xscale='log', xlabel='work (dc)', ylabel='recall', title='recall vs work (envelope; dotted = floor)')
    ax[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG / f'{ds}_curves.png')
    plt.show()

## 3. Price list: loss ratios and group shares

The water-filling allocator takes cuts from the left of the sorted loss list first. A wide spread between groups is what lets it protect hard queries.

In [ ]:
for ds, (pl, sweep) in cal.items():
    steps = pd.DataFrame(pl['steps_sorted'])
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
    if len(steps):
        pos = steps[steps.loss > 0]
        if len(pos):
            bins = np.logspace(np.log10(pos.loss.min()), np.log10(pos.loss.max()) + 1e-9, 30)
            for g, s in pos.groupby('group'):
                ax[0].hist(s.loss, bins=bins, alpha=0.6, label=f'g{g}')
        ax[0].set(xscale='log', xlabel='loss ℓ = Δrecall / Δwork', ylabel='# cut steps', title=f'{ds}: loss ratios')
        ax[0].legend(fontsize=7)
        # cumulative expected saving along the sorted list (what water_level() walks)
        p = {g['id']: g['p'] for g in pl['groups']}
        saved = np.cumsum([p[g] * w for g, w in zip(steps.group, steps.d_work)])
        ax[1].plot(steps.loss.clip(lower=1e-12), pl['nominal_work_mean'] - saved, drawstyle='steps-post')
        ax[1].axhline(pl['floor_work_mean'], color='r', ls=':', label='floor c(k)')
        ax[1].set(xscale='log', xlabel='water level λ*', ylabel='expected work per query (dc)',
                  title='demand left after taking cuts with ℓ ≤ λ*')
        ax[1].legend(fontsize=7)
    ax[2].bar([f"g{g['id']}" for g in pl['groups']], [g['p'] for g in pl['groups']])
    for i, g in enumerate(pl['groups']):
        ax[2].text(i, g['p'], f"e*={g['ef_pref_median']:.0f}", ha='center', va='bottom', fontsize=7)
    ax[2].set(ylabel='share p_g', title='group shares (median preferred ef)')
    fig.tight_layout()
    fig.savefig(FIG / f'{ds}_price_list.png')
    plt.show()

## 4. Ada-ef at its own operating point

Distribution of Ada-ef's preferred ef and of per-query recall at that ef (the tail metrics P5/P1 the Ada-ef paper reports), plus the probe share of the work.

In [ ]:
for ds, (pl, sweep) in cal.items():
    pref = sweep[sweep.is_pref]
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
    ax[0].hist(np.log2(pref.ef_pref.clip(lower=1)), bins=30)
    ax[0].set(xlabel='log2(ef_pref)', ylabel='# queries', title=f'{ds}: Ada-ef ef')
    ax[1].hist(pref.recall, bins=np.linspace(0, 1, pl['k'] + 2))
    ax[1].axvline(pl['target_recall'], color='k', ls='--')
    ax[1].set(xlabel='recall@k at ef_pref', title=f"mean {pref.recall.mean():.3f}  P5 {np.percentile(pref.recall, 5):.2f}  P1 {np.percentile(pref.recall, 1):.2f}")
    ax[2].scatter(pref.score, pref.ef_pref, s=4, alpha=0.4)
    for g in pl['groups']:
        ax[2].axvline(g['score_lo'], color='grey', lw=0.5)
    ax[2].set(yscale='log', xlabel='Ada-ef score', ylabel='ef_pref', title='score → ef (grey = group edges)')
    ax[3].hist(pref.probe_ndis / pref.ndis.clip(lower=1), bins=30)
    ax[3].set(xlabel='probe_ndis / ndis', title='probe share of the work')
    fig.tight_layout()
    fig.savefig(FIG / f'{ds}_operating_point.png')
    plt.show()

## 5. Static ef needed for the target (baseline)

Mean recall over all calibration queries per ladder rung; `static_ef_for_target` is the first rung at or above the target.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for ds, (pl, sweep) in cal.items():
    m = sweep[~sweep.is_pref].groupby('ef')['recall'].mean()
    ax.plot(m.index, m.values, '-o', ms=3, label=f"{ds} (static ef={pl['static_ef_for_target']})")
if cal:
    ax.axhline(next(iter(cal.values()))[0]['target_recall'], color='k', ls='--', lw=1)
ax.set(xscale='log', xlabel='static ef', ylabel='mean recall', title='static-ef recall curve')
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG / 'static_ef.png')
plt.show()